In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Clone your repository
!git clone https://github.com/DevGurav/fall-detect-system.git /content/fall-detect-system

# Create the data directory
!mkdir -p /content/fall-detect-system/ml/data/raw

# Unzip BOTH datasets directly from your Drive into the Colab raw folder
!unzip -q /content/drive/MyDrive/FallGuardian/WEDA-FALL-main.zip -d /content/fall-detect-system/ml/data/raw/
!unzip -q /content/drive/MyDrive/FallGuardian/SmartFallMM-Dataset-main.zip -d /content/fall-detect-system/ml/data/raw/

# Install the required Python packages
!pip install -q torch onnx pydantic structlog

Cloning into '/content/fall-detect-system'...
remote: Enumerating objects: 1528, done.
remote: Counting objects: 100% (417/417), done.
remote: Compressing objects: 100% (292/292), done.
remote: Total 1528 (delta 162), reused 330 (delta 105), pack-reused 1111 (from 1)
Receiving objects: 100% (1528/1528), 43.24 MiB | 37.94 MiB/s, done.
Resolving deltas: 100% (620/620), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.8 MB/s eta 0:00:00


In [ ]:
import os

# Move into the machine learning directory
os.chdir('/content/fall-detect-system/ml')

# Tell Python to look inside the 'src' folder for your custom AI packages
os.environ['PYTHONPATH'] = '/content/fall-detect-system/ml/src'

# FIX: Removed the extra '/dataset' from the end of the WEDA root path!
os.environ['FG_WEDA_ROOT'] = '/content/fall-detect-system/ml/data/raw/WEDA-FALL-main'
os.environ['FG_SMARTFALL_ROOT'] = '/content/fall-detect-system/ml/data/raw/SmartFallMM-Dataset-main'
os.environ['FG_CLOUD_ARTIFACT_DIR'] = '/content/fall-detect-system/ml/artifacts/cloud'
os.environ['FG_BACKEND_MODEL_DIR'] = '/content/fall-detect-system/backend/app/model'
os.environ['FG_MLFLOW'] = '0' # Disable MLflow logging for Colab

# Run the script!
!python -m fall_guardian_ml.training.cross_validate

[data] WEDA-FALL : 8407 windows | 700 fall (8.3%) | 5573 ADL-sourced | 25 subjects [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
/content/fall-detect-system/ml/src/fall_guardian_ml/datasets/smartfall_adl.py:137: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(df["time"], errors="coerce")
/content/fall-detect-system/ml/src/fall_guardian_ml/datasets/smartfall_adl.py:137: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(df["time"], errors="coerce")
/content/fall-detect-system/ml/src/fall_guardian_ml/datasets/smartfall_adl.py:137: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`

In [ ]:
!pip install onnxscript onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.0 MB/s eta 0:00:00


In [ ]:
import sys
import os
import torch

# 1. Add your repository path to Python so it can find your model architecture
sys.path.append("/content/fall-detect-system/ml/src")

# 2. Locate your saved checkpoint and destination directory
# Note: Check your Colab file explorer to confirm the exact name of your checkpoint directory if this differs
checkpoint_dir = "/content/fall-detect-system/ml/checkpoints"
backend_model_dir = "/content/fall-detect-system/backend/app/model"
os.makedirs(backend_model_dir, exist_ok=True)

# Find the best checkpoint file (.pt or .ckpt) generated by your training
checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith(('.pt', '.ckpt'))]

if not checkpoints:
    print("❌ No checkpoint files found in checkponts directory. Double check your file explorer paths.")
else:
    best_checkpoint = os.path.join(checkpoint_dir, sorted(checkpoints)[-1])
    print(f"Found checkpoint: {best_checkpoint}")

    # 3. Import your specific export function directly from your training script
    from fall_guardian_ml.training.cross_validate import export_onnx

    # Mock metadata for the exporter
    cv_meta = {"status": "recovered"}

    print("🔄 Attempting to export checkpoint to ONNX...")
    try:
        onnx_path = export_onnx(best_checkpoint, backend_model_dir, cv_meta)
        print(f"🎉 Success! Your ONNX model has been saved to: {backend_model_dir}/cloud_detector.onnx")
    except Exception as e:
        print(f"❌ Standard exporter failed: {e}")
        print("Trying direct PyTorch export fallback...")

        # Fallback trace in case the custom function requires complex metadata
        model = torch.load(best_checkpoint)
        model.eval()
        dummy_input = torch.randn(1, 50, 6) # Adjust dimensions if your window size/features are different
        output_onnx = os.path.join(backend_model_dir, "cloud_detector.onnx")
        torch.onnx.export(model, dummy_input, output_onnx, export_params=True, opset_version=14)
        print(f"🎉 Fallback Success! Model exported to {output_onnx}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/fall-detect-system/ml/checkpoints'

In [ ]:
import sys
import os
import torch

# 1. Add repository path to Python paths so it can import the model architecture
sys.path.append("/content/fall-detect-system/ml/src")

# 2. Automatically search the entire environment for your checkpoint files
def find_best_checkpoint(search_path="/content"):
    target_files = []
    for root, dirs, files in os.walk(search_path):
        # Ignore the target directory so we don't accidentally select an old export
        if "backend/app/model" in root:
            continue
        for file in files:
            if file.endswith(('.pt', '.ckpt', '.pth')):
                target_files.append(os.path.join(root, file))
    # Return the most recently modified or highest sorted checkpoint file
    return sorted(target_files)[-1] if target_files else None

# Locate the files
best_checkpoint = find_best_checkpoint()
backend_model_dir = "/content/fall-detect-system/backend/app/model"
os.makedirs(backend_model_dir, exist_ok=True)

if not best_checkpoint:
    print("❌ Could not find any saved checkpoint files (.pt, .ckpt, .pth) anywhere inside /content.")
    print("Look at the file tree on the left panel of Colab and see where your script saved its output.")
else:
    print(f"🎯 Automatically located checkpoint at: {best_checkpoint}")

    from fall_guardian_ml.training.cross_validate import export_onnx
    cv_meta = {"status": "recovered"}

    print("🔄 Exporting to ONNX format...")
    try:
        onnx_path = export_onnx(best_checkpoint, backend_model_dir, cv_meta)
        print(f"🎉 Success! Your ONNX model has been saved to: {backend_model_dir}/cloud_detector.onnx")
    except Exception as e:
        print(f"❌ Standard exporter failed: {e}")
        print("Trying direct PyTorch export fallback...")
        try:
            # Direct tracing fallback if custom export configuration metadata is missing
            model = torch.load(best_checkpoint)
            model.eval()

            # Create a standard dummy tensor matching your window size configuration (Batch, Window, Features)
            dummy_input = torch.randn(1, 50, 6)
            output_onnx = os.path.join(backend_model_dir, "cloud_detector.onnx")

            torch.onnx.export(model, dummy_input, output_onnx, export_params=True, opset_version=14)
            print(f"🎉 Fallback Success! Model exported to {output_onnx}")
        except Exception as fallback_err:
            print(f"❌ Fallback also failed: {fallback_err}")

In [ ]:
from google.colab import files

# Zip the mlruns folder and the mlflow.db file together
!zip -r /content/phase30_mlflow_data.zip /content/fall-detect-system/ml/mlruns /content/fall-detect-system/ml/mlflow.db

# Automatically trigger the download to your laptop
files.download("/content/phase30_mlflow_data.zip")
